# Lab 13: Resource Dashboard & Cost Monitor

## Overview
In this lab you will build a Streamlit dashboard for monitoring AWS resources, tracking costs, viewing Bedrock API usage via CloudTrail, checking OpenSearch Serverless collection health, and running automated prerequisite health checks. This covers the observability, cost governance, and resource management skills tested across Domain 4 of the DEA-C01 exam.

## Learning Objectives
- Query AWS resources by tag using the Resource Groups Tagging API
- Use the Cost Explorer API to retrieve daily spend and project monthly costs
- Analyze CloudTrail events for Bedrock API activity
- Monitor OpenSearch Serverless collection status and indices
- Build automated health checks to validate lab prerequisites

## Exam Domain
**Security, Compliance & Governance (20%)** — This lab covers monitoring and observability for GenAI workloads, cost governance with Cost Explorer and resource tagging, CloudTrail audit logging, and operational health checks — all skills tested in Task 4.2 (implement logging and monitoring) and Task 4.3 (ensure compliance and governance).

## Architecture Diagram
![Lab 13 Architecture](../../assets/diagrams/lab-13-resource-dashboard.png)

In [ ]:
%pip install boto3 streamlit opensearch-py requests-aws4auth --quiet

In [ ]:
import boto3, json, os, time
from datetime import datetime, timedelta
from botocore.exceptions import ClientError

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

sts = session.client("sts")
s3 = session.client("s3")
iam = session.client("iam")
ce = session.client("ce")
cloudtrail = session.client("cloudtrail")
aoss = session.client("opensearchserverless")
bedrock = session.client("bedrock")
tagging = session.client("resourcegroupstaggingapi")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE = "genai-lab-bedrock-role"
LAMBDA_ROLE = "genai-lab-lambda-role"
COLLECTION_NAME = "genai-lab-vectors"

print(f"Account:      {ACCOUNT_ID}")
print(f"Region:       {REGION}")
print(f"S3 bucket:    {BUCKET}")
print(f"Collection:   {COLLECTION_NAME}")

## A. Resource Inventory

The **Resource Groups Tagging API** lets you discover all AWS resources that share a common tag. In our lab guide every resource is tagged with `Project = genai-lab-guide`, so a single API call returns the full inventory. We supplement this with direct checks for resources that may not appear in the tagging API (S3 buckets, IAM roles, OpenSearch collections).

This is the foundation of the **Resource Inventory** tab in the dashboard.

In [ ]:
# Query all resources tagged with Project=genai-lab-guide
paginator = tagging.get_paginator("get_resources")
tagged_resources = []

for page in paginator.paginate(
    TagFilters=[{"Key": "Project", "Values": ["genai-lab-guide"]}]
):
    for r in page.get("ResourceTagMappingList", []):
        arn = r["ResourceARN"]
        tags = {t["Key"]: t["Value"] for t in r.get("Tags", [])}
        tagged_resources.append({"arn": arn, "tags": tags})

print(f"Found {len(tagged_resources)} tagged resources:")
for r in tagged_resources:
    print(f"  {r['arn']}")

In [ ]:
# Direct checks — verify specific resources exist even if tags are missing
print("Direct resource checks:")
print("-" * 50)

# S3 bucket
try:
    s3.head_bucket(Bucket=BUCKET)
    print(f"  S3 bucket '{BUCKET}': EXISTS")
except ClientError:
    print(f"  S3 bucket '{BUCKET}': NOT FOUND")

# IAM roles
for role_name in [BEDROCK_ROLE, LAMBDA_ROLE]:
    try:
        iam.get_role(RoleName=role_name)
        print(f"  IAM role '{role_name}': EXISTS")
    except ClientError:
        print(f"  IAM role '{role_name}': NOT FOUND")

# OpenSearch collection
try:
    resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
    colls = resp.get("collectionDetails", [])
    if colls:
        print(f"  OpenSearch collection '{COLLECTION_NAME}': {colls[0]['status']}")
    else:
        print(f"  OpenSearch collection '{COLLECTION_NAME}': NOT FOUND")
except ClientError as e:
    print(f"  OpenSearch collection check error: {e}")

## B. Cost Tracker

The **Cost Explorer API** (`ce`) provides programmatic access to your AWS spending data. We retrieve daily costs for the last 7 days grouped by service, then calculate a total and projected monthly cost.

**Important:** Cost Explorer must be enabled in the Billing console and can take up to 24 hours to activate. The API will raise `AccessDeniedException` until it is ready.

In [ ]:
# Retrieve 7-day cost breakdown by service
today = datetime.utcnow().date()
start = today - timedelta(days=7)

try:
    result = ce.get_cost_and_usage(
        TimePeriod={
            "Start": start.strftime("%Y-%m-%d"),
            "End": today.strftime("%Y-%m-%d"),
        },
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
        GroupBy=[{"Type": "DIMENSION", "Key": "SERVICE"}],
    )

    service_totals = {}
    daily_totals = {}

    for period in result.get("ResultsByTime", []):
        day = period["TimePeriod"]["Start"]
        day_total = 0.0
        for group in period.get("Groups", []):
            service = group["Keys"][0]
            amount = float(group["Metrics"]["UnblendedCost"]["Amount"])
            service_totals[service] = service_totals.get(service, 0.0) + amount
            day_total += amount
        daily_totals[day] = day_total

    total_spend = sum(service_totals.values())
    projected = (total_spend / 7) * 30

    print(f"Total spend (7 days): ${total_spend:.2f}")
    print(f"Projected monthly:    ${projected:.2f}")
    print()
    print("Daily totals:")
    for day, cost in sorted(daily_totals.items()):
        print(f"  {day}: ${cost:.4f}")
    print()
    print("By service (top 10):")
    for svc, cost in sorted(service_totals.items(), key=lambda x: x[1], reverse=True)[:10]:
        if cost > 0.0001:
            print(f"  {svc}: ${cost:.4f}")

    print()
    print("NOTE: OpenSearch Serverless is your #1 cost — run cleanup-all.py when done.")

except ClientError as e:
    if e.response["Error"]["Code"] == "AccessDeniedException":
        print("Cost Explorer access denied.")
        print("Enable it in the Billing console: https://console.aws.amazon.com/billing/home#/costexplorer")
        print("It can take up to 24 hours to activate.")
    else:
        print(f"Error: {e}")

## C. Bedrock Usage via CloudTrail

**AWS CloudTrail** automatically records every API call made in your account. By filtering for `EventSource = bedrock.amazonaws.com`, we can see all Bedrock API activity — `InvokeModel`, `Converse`, `CreateAgent`, `CreateGuardrail`, and more.

This is distinct from **model invocation logging** (Lab 11), which captures the actual prompt and response content. CloudTrail captures API metadata: who called what, when, and from where.

In [ ]:
# Query CloudTrail for recent Bedrock API calls
events_resp = cloudtrail.lookup_events(
    LookupAttributes=[{
        "AttributeKey": "EventSource",
        "AttributeValue": "bedrock.amazonaws.com",
    }],
    MaxResults=20,
)

events = events_resp.get("Events", [])
print(f"Found {len(events)} recent Bedrock events:\n")

action_counts = {}
for ev in events:
    ts = ev.get("EventTime", "")
    name = ev.get("EventName", "")
    user = ev.get("Username", "—")
    print(f"  {ts}  {name:<40s}  {user}")
    action_counts[name] = action_counts.get(name, 0) + 1

print("\nCalls by API action:")
for action, count in sorted(action_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {action}: {count}")

In [ ]:
# Inspect a sample CloudTrail event in detail
if events:
    sample = json.loads(events[0]["CloudTrailEvent"])
    print("Sample CloudTrail Event Detail")
    print("=" * 50)
    print(f"Event name:     {sample.get('eventName')}")
    print(f"Event time:     {sample.get('eventTime')}")
    print(f"Source IP:      {sample.get('sourceIPAddress')}")
    print(f"User agent:     {sample.get('userAgent', '')[:80]}")
    print(f"User identity:  {json.dumps(sample.get('userIdentity', {}), indent=2)[:300]}")
    print(f"Request params: {json.dumps(sample.get('requestParameters', {}), indent=2)[:300]}")
else:
    print("No Bedrock events found — run an earlier lab to generate activity.")

## D. OpenSearch Serverless Status

The OpenSearch Serverless collection `genai-lab-vectors` is used by Labs 04 and 05 for vector search and RAG. Since it runs 24/7 and is the most expensive resource in this lab guide, monitoring its status is critical.

We use `batch_get_collection` to check the collection status, then optionally connect with `opensearch-py` to list indices and document counts.

In [ ]:
# Check OpenSearch collection status
try:
    resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
    collections = resp.get("collectionDetails", [])

    if collections:
        coll = collections[0]
        print(f"Collection: {coll['name']}")
        print(f"Status:     {coll.get('status', '—')}")
        print(f"ARN:        {coll.get('arn', '—')}")
        print(f"Endpoint:   {coll.get('collectionEndpoint', '—')}")
        print(f"Created:    {coll.get('createdDate', '—')}")
        print(f"Modified:   {coll.get('lastModifiedDate', '—')}")
    else:
        errors = resp.get("collectionErrorDetails", [])
        if errors:
            print(f"Collection not found: {errors[0].get('errorMessage', '—')}")
        else:
            print(f"Collection '{COLLECTION_NAME}' does not exist.")
            print("Run scripts/setup-resources.py to create it.")

except ClientError as e:
    print(f"Error: {e}")

In [ ]:
# Connect to OpenSearch and list indices (requires collection to be ACTIVE)
try:
    from opensearchpy import OpenSearch, RequestsHttpConnection
    from requests_aws4auth import AWS4Auth

    resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
    collections = resp.get("collectionDetails", [])

    if collections and collections[0].get("collectionEndpoint"):
        endpoint = collections[0]["collectionEndpoint"]
        host = endpoint.replace("https://", "")

        credentials = session.get_credentials().get_frozen_credentials()
        auth = AWS4Auth(
            credentials.access_key,
            credentials.secret_key,
            REGION,
            "aoss",
            session_token=credentials.token,
        )

        client = OpenSearch(
            hosts=[{"host": host, "port": 443}],
            http_auth=auth,
            use_ssl=True,
            verify_certs=True,
            connection_class=RequestsHttpConnection,
            timeout=10,
        )

        indices = client.cat.indices(format="json")
        if indices:
            print(f"Found {len(indices)} index/indices:\n")
            for idx in indices:
                print(f"  Index:  {idx.get('index', '—')}")
                print(f"  Health: {idx.get('health', '—')}")
                print(f"  Docs:   {idx.get('docs.count', '—')}")
                print(f"  Size:   {idx.get('store.size', '—')}")
                print()
        else:
            print("No indices found in the collection.")
    else:
        print("Collection endpoint not available — collection may still be creating.")

except ImportError:
    print("Install opensearch-py and requests-aws4auth:")
    print("  pip install opensearch-py requests-aws4auth")
except Exception as e:
    print(f"Could not connect to OpenSearch: {e}")

## E. Health Check

This section validates that all lab prerequisites are in place — equivalent to the `scripts/check-prerequisites.sh` script but implemented in Python. Each check reports pass/fail with a specific fix action.

In the Streamlit dashboard, these appear as green checkmarks or red error messages.

In [ ]:
# Run all prerequisite health checks
checks_passed = 0
checks_failed = 0

def check(name, passed, fix=""):
    global checks_passed, checks_failed
    if passed:
        checks_passed += 1
        print(f"  PASS  {name}")
    else:
        checks_failed += 1
        msg = f"  FAIL  {name}"
        if fix:
            msg += f" — fix: {fix}"
        print(msg)

print("Health Check Results")
print("=" * 60)

# 1. AWS credentials
try:
    sts.get_caller_identity()
    check("AWS credentials valid", True)
except Exception:
    check("AWS credentials valid", False, "run 'aws configure'")

# 2. S3 bucket
try:
    s3.head_bucket(Bucket=BUCKET)
    check(f"S3 bucket '{BUCKET}' exists", True)
except ClientError:
    check(f"S3 bucket '{BUCKET}' exists", False, "run scripts/setup-resources.py")

# 3. IAM roles
for role in [BEDROCK_ROLE, LAMBDA_ROLE]:
    try:
        iam.get_role(RoleName=role)
        check(f"IAM role '{role}' exists", True)
    except ClientError:
        check(f"IAM role '{role}' exists", False, "run scripts/setup-resources.py")

# 4. OpenSearch collection ACTIVE
try:
    resp = aoss.batch_get_collection(names=[COLLECTION_NAME])
    colls = resp.get("collectionDetails", [])
    if colls and colls[0].get("status") == "ACTIVE":
        check(f"OpenSearch collection '{COLLECTION_NAME}' is ACTIVE", True)
    elif colls:
        check(f"OpenSearch collection '{COLLECTION_NAME}' is ACTIVE", False,
              f"status is {colls[0].get('status')} — wait for ACTIVE")
    else:
        check(f"OpenSearch collection '{COLLECTION_NAME}' exists", False,
              "run scripts/setup-resources.py")
except ClientError:
    check(f"OpenSearch collection check", False, "check permissions")

# 5. Bedrock model access
try:
    models = bedrock.list_foundation_models()
    count = len(models.get("modelSummaries", []))
    check(f"Bedrock model access ({count} models)", count > 0,
          "enable model access in Bedrock console")
except ClientError:
    check("Bedrock model access", False, "check IAM permissions")

print("=" * 60)
print(f"Results: {checks_passed} passed, {checks_failed} failed")

## F. Run the Dashboard

The sections above demonstrate each panel individually. The complete dashboard combines all five panels into a tabbed Streamlit application.

To launch the dashboard, open a terminal in the `labs/13-resource-dashboard/` directory and run:

```bash
streamlit run dashboard.py
```

This will open a browser tab with the full multi-tab dashboard. You can also run it from the project root:

```bash
streamlit run labs/13-resource-dashboard/dashboard.py
```

The dashboard refreshes on each interaction. Use the **Refresh** button in the sidebar to pull the latest data.

## Key Takeaways

1. **The Resource Groups Tagging API provides a single-query inventory of all resources sharing a tag** — by consistently tagging every resource with `Project = genai-lab-guide`, you can discover your full infrastructure footprint without knowing resource types or ARNs in advance
2. **Cost Explorer's `get_cost_and_usage` API enables programmatic cost monitoring with daily granularity** — grouping by SERVICE reveals which AWS services dominate your spend, and projecting a 7-day window to 30 days gives early warning of budget overruns
3. **CloudTrail records API metadata (who, what, when) for every Bedrock call automatically** — filtering by `EventSource = bedrock.amazonaws.com` provides an audit trail without any additional configuration, complementing model invocation logging which captures prompt/response content
4. **OpenSearch Serverless collections run 24/7 and are the most significant cost in this lab guide** — monitoring collection status programmatically and cleaning up promptly after completing labs is essential for cost governance
5. **Automated health checks replace manual verification and catch configuration drift** — checking credentials, buckets, roles, collections, and model access in code ensures prerequisites are always validated before running labs

## Key Concepts

| Concept | Definition |
|---------|------------|
| Resource Groups Tagging API | An AWS API that queries resources across services by tag key-value pairs, enabling inventory discovery and cost allocation without service-specific API calls — returns ARNs for all matching resources in a single paginated response |
| Cost Explorer | An AWS service and API (`ce`) that provides detailed cost and usage data, supporting daily or monthly granularity with grouping by service, tag, region, or other dimensions — must be enabled in the Billing console before API access works |
| CloudTrail | An AWS audit logging service that records every API call across your account — captures metadata (caller identity, timestamp, source IP, request parameters) but not model prompt/response content, which requires separate Bedrock invocation logging |
| Invocation Logging | A Bedrock-specific logging feature that captures the actual input prompts and model responses, delivered to CloudWatch Logs and/or S3 — distinct from CloudTrail which only records API call metadata without content |
| Resource Tagging | The practice of attaching key-value metadata (e.g., `Project`, `Environment`, `Team`) to AWS resources for cost allocation, access control, automation, and inventory management — tags can be enforced via SCPs or IAM condition keys |
| CloudWatch | An AWS monitoring service that collects metrics, logs, and events from AWS resources — Bedrock invocation logs and custom application metrics can be sent to CloudWatch for dashboarding, alerting, and anomaly detection |
| Health Check | An automated validation pattern that programmatically verifies infrastructure prerequisites (credentials, resources, permissions, connectivity) before executing workloads — reduces debugging time by catching missing or misconfigured dependencies early |

## Exam Preparation — Common Exam Question Patterns

**Q: A company wants to monitor which Bedrock models are being invoked and by whom, without capturing the actual prompt content. What AWS service should they use?**
A: AWS CloudTrail. CloudTrail automatically records all Bedrock API calls including `InvokeModel` and `Converse`, capturing the caller identity, timestamp, source IP, and request parameters (such as `modelId`). CloudTrail does NOT capture prompt or response content — that requires enabling Bedrock model invocation logging separately via `put_model_invocation_logging_configuration`.

**Q: How can an organization discover all AWS resources associated with a specific GenAI project across multiple services?**
A: Use the Resource Groups Tagging API with a tag filter. If all project resources are consistently tagged (e.g., `Project = genai-lab-guide`), calling `get_resources` with that tag filter returns ARNs for every matching resource across S3, IAM, Bedrock, OpenSearch, and other services in a single paginated response. This approach requires disciplined tagging at resource creation time.

**Q: A team is surprised by high AWS costs after running GenAI labs. What programmatic approach can they use to identify the most expensive services?**
A: Use the Cost Explorer API (`get_cost_and_usage`) with `GroupBy=[{"Type": "DIMENSION", "Key": "SERVICE"}]` to retrieve daily cost breakdowns by service. This immediately reveals which services dominate spend (typically OpenSearch Serverless for vector search workloads). Combining this with AWS Budgets for proactive alerts and resource tagging for cost allocation enables comprehensive cost governance.

**Q: What is the difference between CloudTrail logging and Bedrock model invocation logging for compliance purposes?**
A: CloudTrail records API metadata — who called which Bedrock API, when, from what IP address, and with what request parameters. Model invocation logging captures the actual content — input prompts, model responses, and token usage. For security auditing (detecting unauthorized access), CloudTrail is sufficient. For content compliance (ensuring no sensitive data is sent to models, reviewing model outputs for policy violations), invocation logging is required. Both should be enabled for comprehensive compliance.

**Q: How should a company validate that their GenAI infrastructure is correctly configured before deploying a new application?**
A: Implement automated health checks that programmatically verify each prerequisite: (1) valid AWS credentials via `sts:GetCallerIdentity`, (2) required S3 buckets via `s3:HeadBucket`, (3) IAM roles via `iam:GetRole`, (4) OpenSearch collection status via `batch_get_collection`, (5) Bedrock model access via `list_foundation_models`. Run these checks as part of the deployment pipeline, failing fast with specific remediation instructions for each missing dependency.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Cost Explorer API — Section B | ~1 API call | Free (included in account) |
| CloudTrail lookup — Section C | ~2 API calls | Free (90-day event history) |
| OpenSearch Serverless — Section D (read-only) | Existing collection, no new resources | $0.00 |
| Streamlit dashboard — Section F | Local process, no AWS cost | $0.00 |
| **Total** | | **~$0.00 (read-only lab)** |

This lab is read-only — it queries existing resources but does not create new ones. The only cost is from the resources created by previous labs (primarily OpenSearch Serverless). Run `scripts/cleanup-all.py` when you are finished with all labs.